In [1]:
!pip install mediapipe==0.10.14 Pillow==10.4.0 torchvision -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 117.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 21.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
a2a-sdk 0.3.23 requires protobuf>=5.29.5, but you have protobuf 4.25.8 which is incompatible.
grain 0.2.15 requires protobuf>=5.28.3, but you have protobuf 4.25.8 which is incompatible.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.8 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
ydf 0.14.0 requires p

In [2]:
import cv2
import mediapipe as mp
import os
import numpy as np
import math
import random
from tqdm import tqdm

2026-03-19 14:21:31.077611: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773930091.264802      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773930091.318775      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773930091.754832      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773930091.754869      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773930091.754871      24 computation_placer.cc:177] computation placer alr

In [3]:
# 0. CONFIG - BẢNG ĐIỀU KHIỂN HYPERPARAMETERS (Cập nhật từ Histogram)
MAR_YAWN_THRESH = 0.35       # Hạ ngưỡng xuống 0.35 để bắt được Early Yawn
MAR_NO_YAWN_UPPER = 1.0      # Mở khóa trần để lấy toàn bộ các pha cười to/nói lớn (Hard Negatives)
MAR_NO_YAWN_LOWER = 0.15     # Ngưỡng dưới: Bỏ qua phần lớn các pha ngậm miệng tĩnh lặng
TEMPORAL_FRAMES = 3          # Cần 3 frame liên tiếp thỏa mãn để xác nhận 1 chuỗi ngáp
NO_YAWN_SAMPLE_RATE = 0.2    # Lấy 20% các frame tĩnh lặng để cân bằng dữ liệu
FRAME_SKIP = 2               

ROOT_DIR = '/kaggle/input/datasets/enider/yawdd-dataset' 
OUTPUT_DIR = '/kaggle/working/YawDD_Ultimate_V4'

for split in ['train', 'val']:
    os.makedirs(os.path.join(OUTPUT_DIR, split, 'yawn'), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, split, 'no_yawn'), exist_ok=True)

mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(static_image_mode=False, max_num_faces=1, refine_landmarks=False)
MOUTH_IDXS = [61, 291, 0, 17, 39, 40, 37, 267, 269, 270, 409, 287, 375, 321, 405, 314, 84, 181, 91, 146]

def calculate_mar(landmarks, w, h):
    p_left = (landmarks.landmark[61].x * w, landmarks.landmark[61].y * h)
    p_right = (landmarks.landmark[291].x * w, landmarks.landmark[291].y * h)
    p_top = (landmarks.landmark[13].x * w, landmarks.landmark[13].y * h)
    p_bottom = (landmarks.landmark[14].x * w, landmarks.landmark[14].y * h)
    
    dist_h = math.hypot(p_right[0] - p_left[0], p_right[1] - p_left[1])
    dist_v = math.hypot(p_bottom[0] - p_top[0], p_bottom[1] - p_top[1])
    return dist_v / dist_h if dist_h > 0 else 0.0

# THUẬT TOÁN ADAPTIVE PADDING (Lấy 20% kích thước miệng làm viền)
def get_mouth_crop_adaptive(frame, landmarks, w, h):
    coords = np.array([[int(landmarks.landmark[i].x * w), int(landmarks.landmark[i].y * h)] for i in MOUTH_IDXS])
    x_min, y_min = np.min(coords, axis=0)
    x_max, y_max = np.max(coords, axis=0)
    
    box_w = x_max - x_min
    box_h = y_max - y_min
    
    pad_x = int(0.2 * box_w)
    pad_y = int(0.2 * box_h)
    
    x_min, y_min = max(0, x_min - pad_x), max(0, y_min - pad_y)
    x_max, y_max = min(w, x_max + pad_x), min(h, y_max + pad_y)
    return frame[y_min:y_max, x_min:x_max]

# TÁCH TRAIN/VAL THEO VIDEO (Chống Data Leakage)
all_videos = []
for root, dirs, files in os.walk(ROOT_DIR):
    for file in files:
        if file.lower().endswith('.avi'):
            all_videos.append(os.path.join(root, file))

random.seed(42)
random.shuffle(all_videos)
split_idx = int(0.8 * len(all_videos))
train_videos = set(all_videos[:split_idx])
val_videos = set(all_videos[split_idx:])

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1773930109.869426      78 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1773930109.879213      78 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [4]:
# 1. BẮT ĐẦU VÒNG LẶP LỌC DỮ LIỆU
for video_path in tqdm(all_videos, desc="Đang trích xuất Data sạch..."):
    video_name = os.path.basename(video_path)
    base_name = os.path.splitext(video_name)[0]
    phase_dir = 'train' if video_path in train_videos else 'val'
    
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    yawn_buffer = [] 
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        
        frame_count += 1
        if frame_count % FRAME_SKIP != 0: continue
            
        h, w, _ = frame.shape
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(rgb_frame)
        
        if results.multi_face_landmarks:
            for face_landmarks in results.multi_face_landmarks:
                mar_value = calculate_mar(face_landmarks, w, h)
                
                # NẾU LÀ VIDEO NGÁP
                if 'yawn' in video_name.lower():
                    if mar_value > MAR_YAWN_THRESH:
                        mouth_img = get_mouth_crop_adaptive(frame, face_landmarks, w, h)
                        if mouth_img.size > 0:
                            mouth_img_resized = cv2.resize(mouth_img, (224, 224))
                            save_path = os.path.join(OUTPUT_DIR, phase_dir, 'yawn', f"{base_name}_{frame_count}.jpg")
                            yawn_buffer.append((mouth_img_resized, save_path))
                    else:
                        # Kết thúc chuỗi ngáp -> Kiểm tra độ dài buffer và xả ra ổ cứng
                        if len(yawn_buffer) >= TEMPORAL_FRAMES:
                            for img, p in yawn_buffer:
                                # Giảm Redundancy: Chỉ giữ 60% số khung hình trong 1 chuỗi ngáp
                                if random.random() < 0.6: 
                                    cv2.imwrite(p, img)
                        
                        yawn_buffer = [] # Reset buffer
                        
                        # Xử lý vùng xám (Hard Negatives) trong video ngáp
                        if MAR_NO_YAWN_LOWER < mar_value <= MAR_NO_YAWN_UPPER:
                            if random.random() < NO_YAWN_SAMPLE_RATE:
                                mouth_img = get_mouth_crop_adaptive(frame, face_landmarks, w, h)
                                if mouth_img.size > 0:
                                    mouth_img_resized = cv2.resize(mouth_img, (224, 224))
                                    save_path = os.path.join(OUTPUT_DIR, phase_dir, 'no_yawn', f"{base_name}_{frame_count}.jpg")
                                    cv2.imwrite(save_path, mouth_img_resized)

                # NẾU LÀ VIDEO BÌNH THƯỜNG (Cười, nói, im lặng)
                else: 
                    # Lấy mạnh các pha há miệng to (Hard Negatives), lấy mỏng các pha ngậm miệng (Im lặng)
                    sample_prob = 1.0 if mar_value > MAR_NO_YAWN_LOWER else NO_YAWN_SAMPLE_RATE
                    if random.random() < sample_prob:
                        mouth_img = get_mouth_crop_adaptive(frame, face_landmarks, w, h)
                        if mouth_img.size > 0:
                            mouth_img_resized = cv2.resize(mouth_img, (224, 224))
                            save_path = os.path.join(OUTPUT_DIR, phase_dir, 'no_yawn', f"{base_name}_{frame_count}.jpg")
                            cv2.imwrite(save_path, mouth_img_resized)

    # Xả nốt buffer nếu video kết thúc khi đang ngáp dở
    if len(yawn_buffer) >= TEMPORAL_FRAMES:
        for img, p in yawn_buffer:
            if random.random() < 0.6:
                cv2.imwrite(p, img)
                
    cap.release()

Đang trích xuất Data sạch...:   0%|          | 0/348 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
Đang trích xuất Data sạch...: 100%|██████████| 348/348 [15:37<00:00,  2.69s/it]


In [5]:
# 2. IN THỐNG KÊ DATASET ĐỂ KIỂM SOÁT IMBALANCE
print("\n📊 THỐNG KÊ DỮ LIỆU ĐÃ TRÍCH XUẤT:")
for split in ['train', 'val']:
    y_count = len(os.listdir(os.path.join(OUTPUT_DIR, split, 'yawn')))
    n_y_count = len(os.listdir(os.path.join(OUTPUT_DIR, split, 'no_yawn')))
    ratio = n_y_count / y_count if y_count > 0 else 0
    print(f"👉 Tập {split.upper()}: Yawn = {y_count} ảnh | No Yawn = {n_y_count} ảnh | Tỷ lệ = 1 : {ratio:.2f}")

print("\n🔥 HOÀN TẤT! Bộ dữ liệu đã sẵn sàng để train.")


📊 THỐNG KÊ DỮ LIỆU ĐÃ TRÍCH XUẤT:
👉 Tập TRAIN: Yawn = 1901 ảnh | No Yawn = 33746 ảnh | Tỷ lệ = 1 : 17.75
👉 Tập VAL: Yawn = 440 ảnh | No Yawn = 8524 ảnh | Tỷ lệ = 1 : 19.37

🔥 HOÀN TẤT! Bộ dữ liệu đã sẵn sàng để train.
